In [1]:
!pip install -q anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 23.3 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from anthropic import Anthropic

CLAUDE_API_KEY = ""

client = Anthropic(
    api_key=CLAUDE_API_KEY
)

print("Claude client initialized.")

Claude client initialized.


In [4]:
# ============================================================
# CELL 1 — CONFIG
# ============================================================

import os
import json
import re
import time
from collections import defaultdict

import pandas as pd

CV_RUN = 4   # change to 2,3,4,5 for other CV runs

DATA_PATH = "/content/drive/MyDrive/conversation_en.json"
SPLIT_DIR = "/content/drive/MyDrive/5cv_splits"

TRAIN_IDS_PATH = f"{SPLIT_DIR}/cv{CV_RUN}_train_ids.csv"
VAL_IDS_PATH   = f"{SPLIT_DIR}/cv{CV_RUN}_val_ids.csv"
TEST_IDS_PATH  = f"{SPLIT_DIR}/cv{CV_RUN}_test_ids.csv"

OUTPUT_DIR = f"/content/drive/MyDrive/5cv_augmentation/cv{CV_RUN}"
os.makedirs(OUTPUT_DIR, exist_ok=True)

AUGMENT_LABELS = ["Chitchat", "Revise"]

TARGET_SYNTHETIC_PER_CLASS = {
    "Chitchat": 100,
    "Revise": 100
}

print("CV run:", CV_RUN)
print("Train IDs:", TRAIN_IDS_PATH)

CV run: 4
Train IDs: /content/drive/MyDrive/5cv_splits/cv4_train_ids.csv


In [5]:
# ============================================================
# CELL 2 — LOAD FROZEN CV IDs
# ============================================================

train_cids = set(
    pd.read_csv(TRAIN_IDS_PATH)["conv_id"].astype(str)
)

val_cids = set(
    pd.read_csv(VAL_IDS_PATH)["conv_id"].astype(str)
)

test_cids = set(
    pd.read_csv(TEST_IDS_PATH)["conv_id"].astype(str)
)

print("Train conversations:", len(train_cids))
print("Validation conversations:", len(val_cids))
print("Test conversations:", len(test_cids))

assert len(train_cids) == 528
assert len(val_cids) == 132
assert len(test_cids) == 165

assert train_cids.isdisjoint(val_cids)
assert train_cids.isdisjoint(test_cids)
assert val_cids.isdisjoint(test_cids)

print("PASS: frozen CV split loaded.")

Train conversations: 528
Validation conversations: 132
Test conversations: 165
PASS: frozen CV split loaded.


In [6]:
# ============================================================
# CELL 3 — LOAD RAW DATA AND KEEP CV TRAIN CONVERSATIONS ONLY
# ============================================================

with open(DATA_PATH, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

cv_train_conversations = [
    conv
    for conv in raw_data
    if str(conv["conv_id"]) in train_cids
]

print(
    f"CV{CV_RUN} training conversations recovered:",
    len(cv_train_conversations)
)

assert len(cv_train_conversations) == 528

CV4 training conversations recovered: 528


In [7]:
# ============================================================
# CELL 4 — BUILD AUGMENTATION SOURCE POOL
# Chitchat + Revise ONLY
# ============================================================

augmentation_sources = []

for conv in cv_train_conversations:

    conv_id = str(conv["conv_id"])
    conversation = conv["conversation"]

    for i, msg in enumerate(conversation):

        if msg.get("role") != "user":
            continue

        intent = msg.get("intent")

        if intent not in AUGMENT_LABELS:
            continue

        # Last system message before this target user turn
        previous_system_text = ""

        for previous_msg in reversed(conversation[:i]):
            if previous_msg.get("role") == "system":
                previous_system_text = previous_msg.get(
                    "content", ""
                )
                break

        augmentation_sources.append({
            "conversation_id": conv_id,
            "turn_index": i,
            "system_text": previous_system_text,
            "user_text": msg.get("content", ""),
            "label": intent
        })

source_df = pd.DataFrame(augmentation_sources)

print(f"CV{CV_RUN} augmentation source distribution:")
print(source_df["label"].value_counts())

CV4 augmentation source distribution:
label
Chitchat    179
Revise       48
Name: count, dtype: int64


In [8]:
# ============================================================
# CELL 5 — SAME AUGMENTATION PROMPT
# ============================================================

def build_augmentation_prompt(system_text, user_text, label):

    return f"""
You are an expert in e-commerce conversational AI and dialogue understanding.

Your task is to generate realistic, label-preserving variations of a conversation sample for intent classification training.

STRICT RULES:
- The intent label MUST remain EXACTLY the same.
- Do NOT change the user's goal or meaning.
- Do NOT introduce a different intent.
- Keep language natural, casual, and realistic.
- Do NOT make the text overly formal.
- Preserve conversational style, including short or imperfect phrasing when appropriate.
- If a variation changes the intent, do not generate it.

INPUT
System: "{system_text}"
User: "{user_text}"
Intent Label: "{label}"

Generate exactly 3 variations:

Variation A - User-only
- Remove system message completely
- Rewrite only the user message
- Intent must still be clear

Variation B - Natural Conversation
- Slightly paraphrase the system message
- Slightly rewrite the user response
- Keep semantics identical

Variation C - Abstracted Context
- Replace system message with a short tag like [Clarify], [Recommend], [Confirm], or [Filter]
- Keep user response natural

Return ONLY raw valid JSON in this exact format.
Do not use markdown.
Do not use code fences.

[
  {{
    "type": "User-only",
    "input": "User: <text>",
    "label": "{label}"
  }},
  {{
    "type": "Natural Conversation",
    "input": "System: <text>\\nUser: <text>",
    "label": "{label}"
  }},
  {{
    "type": "Abstracted Context",
    "input": "[Tag]\\nUser: <text>",
    "label": "{label}"
  }}
]
""".strip()

In [9]:
# ============================================================
# CELL 6 — OUTPUT CLEANING
# ============================================================

def clean_json_output(raw_output):

    text = raw_output.strip()

    if text.startswith("```json"):
        text = text[len("```json"):].strip()

    elif text.startswith("```"):
        text = text[len("```"):].strip()

    if text.endswith("```"):
        text = text[:-3].strip()

    return text


def extract_user_text(input_text):

    if not isinstance(input_text, str):
        return ""

    match = re.search(
        r"User:\s*(.*)",
        input_text,
        flags=re.IGNORECASE | re.DOTALL
    )

    if match:
        return match.group(1).strip()

    return input_text.strip()

In [10]:
# ============================================================
# CELL 7 — CLAUDE CALL
# assumes `client` has already been initialized
# ============================================================

def call_claude_for_variations(
    client,
    system_text,
    user_text,
    label
):

    prompt = build_augmentation_prompt(
        system_text,
        user_text,
        label
    )

    response = client.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=600,
        temperature=0.4,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    raw_output = response.content[0].text

    cleaned = clean_json_output(raw_output)

    return json.loads(cleaned)

In [11]:
# ============================================================
# CELL 8 — GENERATE 100 CHITCHAT + 100 REVISE
# ============================================================

synthetic_rows = []

synthetic_counts = {
    "Chitchat": 0,
    "Revise": 0
}

seen = set()

for label in ["Chitchat", "Revise"]:

    label_sources = source_df[
        source_df["label"] == label
    ].reset_index(drop=True)

    source_index = 0

    while (
        synthetic_counts[label]
        < TARGET_SYNTHETIC_PER_CLASS[label]
    ):

        # cycle through training sources if needed
        source_row = label_sources.iloc[
            source_index % len(label_sources)
        ]

        source_index += 1

        try:

            variations = call_claude_for_variations(
                client=client,
                system_text=source_row["system_text"],
                user_text=source_row["user_text"],
                label=label
            )

            for item in variations:

                if (
                    synthetic_counts[label]
                    >= TARGET_SYNTHETIC_PER_CLASS[label]
                ):
                    break

                # Label must remain exactly the same
                if item.get("label") != label:
                    continue

                new_user_text = extract_user_text(
                    item.get("input", "")
                )

                if not new_user_text:
                    continue

                # Reject unchanged copy
                if (
                    new_user_text.strip().lower()
                    ==
                    str(source_row["user_text"])
                    .strip()
                    .lower()
                ):
                    continue

                key = (
                    str(source_row["conversation_id"]),
                    int(source_row["turn_index"]),
                    new_user_text.strip().lower(),
                    label
                )

                if key in seen:
                    continue

                seen.add(key)

                synthetic_rows.append({
                    # provenance kept from original training turn
                    "conversation_id":
                        str(source_row["conversation_id"]),

                    "turn_index":
                        int(source_row["turn_index"]),

                    "role": "user",

                    "text": new_user_text,

                    "intent": label,

                    "is_augmented": True,

                    "augmentation_type":
                        item.get("type", ""),

                    "source_user_text":
                        source_row["user_text"],

                    "source_system_text":
                        source_row["system_text"]
                })

                synthetic_counts[label] += 1

            print(
                f"{label}: "
                f"{synthetic_counts[label]}/"
                f"{TARGET_SYNTHETIC_PER_CLASS[label]}"
            )

            time.sleep(0.5)

        except Exception as e:

            print(
                f"Error | CV{CV_RUN} | "
                f"{label} | source {source_index}: {e}"
            )

            time.sleep(1)

Chitchat: 2/100
Chitchat: 5/100
Chitchat: 7/100
Chitchat: 9/100
Chitchat: 11/100
Chitchat: 14/100
Chitchat: 17/100
Chitchat: 20/100
Chitchat: 23/100
Chitchat: 26/100
Chitchat: 29/100
Chitchat: 32/100
Chitchat: 35/100
Chitchat: 38/100
Chitchat: 41/100
Chitchat: 44/100
Chitchat: 47/100
Chitchat: 50/100
Chitchat: 52/100
Chitchat: 54/100
Chitchat: 57/100
Chitchat: 60/100
Chitchat: 62/100
Chitchat: 64/100
Chitchat: 67/100
Chitchat: 70/100
Chitchat: 73/100
Chitchat: 75/100
Chitchat: 77/100
Chitchat: 79/100
Chitchat: 81/100
Chitchat: 84/100
Chitchat: 86/100
Chitchat: 89/100
Chitchat: 91/100
Chitchat: 93/100
Chitchat: 95/100
Chitchat: 97/100
Chitchat: 99/100
Chitchat: 100/100
Revise: 2/100
Revise: 5/100
Revise: 7/100
Revise: 9/100
Revise: 11/100
Revise: 14/100
Revise: 17/100
Revise: 20/100
Revise: 21/100
Revise: 24/100
Revise: 26/100
Revise: 28/100
Revise: 30/100
Revise: 32/100
Revise: 34/100
Revise: 37/100
Revise: 39/100
Revise: 41/100
Revise: 44/100
Revise: 47/100
Revise: 49/100
Revise: 52/1

In [12]:
# ============================================================
# CELL 9 — VERIFY SYNTHETIC DATA
# ============================================================

synthetic_df = pd.DataFrame(synthetic_rows)

print(
    synthetic_df["intent"].value_counts()
)

print("Total synthetic:", len(synthetic_df))

assert len(synthetic_df) == 200

assert (
    synthetic_df["intent"]
    .value_counts()
    .get("Chitchat", 0)
    == 100
)

assert (
    synthetic_df["intent"]
    .value_counts()
    .get("Revise", 0)
    == 100
)

print("PASS: 100 Chitchat + 100 Revise generated.")

intent
Chitchat    100
Revise      100
Name: count, dtype: int64
Total synthetic: 200
PASS: 100 Chitchat + 100 Revise generated.


In [13]:
# ============================================================
# CELL 10 — AUGMENTATION LEAKAGE CHECK
# ============================================================

synthetic_cids = set(
    synthetic_df[
        "conversation_id"
    ].astype(str)
)

print(
    "Synthetic source conversations:",
    len(synthetic_cids)
)

print(
    "Synthetic ∩ Validation:",
    len(synthetic_cids & val_cids)
)

print(
    "Synthetic ∩ Test:",
    len(synthetic_cids & test_cids)
)

assert synthetic_cids.issubset(train_cids)
assert synthetic_cids.isdisjoint(val_cids)
assert synthetic_cids.isdisjoint(test_cids)

print(
    f"PASS: CV{CV_RUN} augmentation "
    "comes only from frozen training conversations."
)

Synthetic source conversations: 64
Synthetic ∩ Validation: 0
Synthetic ∩ Test: 0
PASS: CV4 augmentation comes only from frozen training conversations.


In [14]:
# ============================================================
# CELL 11 — SAVE
# ============================================================

OUTPUT_PATH = (
    f"{OUTPUT_DIR}/"
    f"cv{CV_RUN}_synthetic_"
    "100_chitchat_100_revise.csv"
)

synthetic_df.to_csv(
    OUTPUT_PATH,
    index=False
)

print("Saved:", OUTPUT_PATH)

Saved: /content/drive/MyDrive/5cv_augmentation/cv4/cv4_synthetic_100_chitchat_100_revise.csv
